# 第 3 周练习 —— 合成数据生成（Synthetic Data Generation）

## 练习目标（理念）

用 **Hugging Face Transformers** 的本地/托管大模型，按你描述的字段结构批量「造」训练或演示用数据：

- 输入：自然语言描述「要什么字段、什么场景」
- 输出：JSON / CSV / JSONL / 纯文本 / 代码测试用例等固定格式
- 交互：用 **Gradio** 搭一个小 UI，选模型与条数后一键生成

## 和本课 Week 3 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Hugging Face `pipeline` | `pipeline("text-generation", ...)` 加载因果语言模型 |
| Chat 模板 | `apply_chat_template` 把 system/user messages 拼成模型输入 |
| 合成数据 | system prompt 约束格式与条数，user prompt 描述业务字段 |
| Gradio Blocks | 下拉框选模型/格式，滑条选条数，按钮触发生成 |

## 怎么跑

1. 先跑安装依赖单元格（`bitsandbytes` / `accelerate` / 固定版 `transformers`）
2. `.env` 里准备 `HF_TOKEN`（部分门禁模型需要）
3. 有 GPU 会走 `device=0`，否则 CPU（`-1`，可能很慢）
4. 启动 Gradio 后，在浏览器里改 prompt、格式与模型再点 Generate


## 能生成哪些格式？

借助 Hugging Face 的**模型**（以及课程里提到的数据集思路），可以创建合成数据，常见落盘格式：

- JSON
- CSV
- JSONL
- 原始文本（Raw / Plain Text）
- 代码（Python 测试用例）

## 示例用户提示（可直接粘到 UI）

- 为产品的客户评论数据集生成合成数据点。字段：`customer_id`、`product_id`、评级、评论。
- 为公司生成员工记录。字段：姓名、电子邮件、电话、部门、工资。
- 生成公司的销售数据记录。字段：`customer_id`、`product_id`、数量、金额。
- 生成城市的天气数据记录。字段：日期、温度、降水量、风速。
- 生成公司股票数据记录。字段：日期、开盘价、最高价、最低价、收盘价。

> 提示字符串本身会作为 **user prompt** 发给模型；下面代码里的 system/format 英文规则**不要翻译**，否则会影响输出格式约束。


In [ ]:
# ========== 安装依赖：量化 / 加速 / 固定 transformers 版本 ==========
# -q：安静安装；--upgrade：尽量升到兼容组合
# transformers==4.57.6：锁定版本，减少「pipeline / chat_template 行为漂移」
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6


In [ ]:
# ========== 导入：后面单元格共用的库 ==========

# 标准库 os：读环境变量（Environment Variables），例如 HF_TOKEN
import os
# typing：给 messages 等结构做类型标注（List / Dict）
from typing import List, Dict
# dotenv：从 .env 加载密钥，避免把 Token 写进笔记本
from dotenv import load_dotenv
# transformers.pipeline：一行加载「文本生成」推理管线
from transformers import pipeline
# Gradio：快速搭 Web UI（Blocks / 组件 / launch）
import gradio as gr
# PyTorch：探测 CUDA，决定 pipeline 的 device
import torch


In [ ]:
# ========== 环境变量：检查 Hugging Face Token 是否就绪 ==========

# override=True：.env 里的值覆盖进程里已有同名环境变量
load_dotenv(override=True)

# 从环境读取 HF_TOKEN（门禁模型 / 较高配额时常需要）
hf_token = os.getenv('HF_TOKEN')
# 有 Token：只打印前 8 位做存在性确认，避免整串泄露到输出
if hf_token:
    print(f"✓ HuggingFace Token exists: {hf_token[:8]}...")
# 没有 Token：提示可能加载失败；URL 保持原样（可运行指引）
else:
    print("✗ HuggingFace Token not set - some models may not work")
    print("  Get your token from https://huggingface.co/settings/tokens")


In [ ]:
# ========== 可选模型列表：UI 下拉框的 choices 来源 ==========
# 每项都是 Hugging Face Hub 上的 model id（字符串必须原样，勿改）
models = [
  "openai/gpt-oss-20b",
  "microsoft/Phi-4-mini-instruct",
  "meta-llama/Llama-3.2-3B-Instruct",
  "Qwen/Qwen3-Coder-Next-FP8"
]


In [ ]:
# ========== 格式规则 + system prompt 拼装 ==========
# FORMAT_RULES：每种输出格式对应一段英文约束（会进 system prompt，禁止翻译）
FORMAT_RULES = {
    "JSON": "Return a valid JSON array. Each element is an object with consistent keys. Start with [ and end with ]. Use double quotes. No markdown fences, no preamble.",
    "CSV": "Return CSV. First line is header with column names. One data row per record. Use commas. Quote fields containing commas.",
    "JSONL": "Return one valid JSON object per line. No blank lines. Each line is self-contained.",
    "Raw Text": "Return prose entries separated by blank lines. Each entry describes one record.",
    "Code": "Return Python code in a singleython fenced block. Use pytest or unittest style test cases."
}



# 按所选 format 与条数，拼出完整的 system 指令（英文模板保持原样）
def build_system_prompt(format: str, record_count: int = 5) -> str:
  # 查表拿格式细则；未知 key 时回退到 JSON 规则
  format_instruction = FORMAT_RULES.get(format, FORMAT_RULES["JSON"])
  # 注意：这里变量名 format 被重新赋值为整段 system 文本（原逻辑如此，勿改）
  format = f"""
  You generate synthetic datasets. Output must be high quality, diverse, and free of PII.
  Constraints:
    - Consistent structure across all records
    - Output ONLY the data—no explanations, no "Here is the JSON/CSV:", no extra text
    - Record count: {record_count}

  Output format: {format}
  Format rules: {format_instruction}
  """
  # 返回拼好的 system content
  return format




# 组装 Chat messages：system 定规则，user 放业务描述
def build_messages(format: str, record_count: int = 5, user_prompt: str = "") -> List[Dict[str, str]]:
  return [
    {"role": "system", "content": build_system_prompt(format, record_count)},
    {"role": "user", "content": user_prompt}
  ]


In [ ]:
# ========== 加载 text-generation 管线 ==========
# model_name：HF Hub 模型 id；返回可用的 pipeline 对象
def load_pipeline(model_name: str):
  # task 固定为文本生成；device：有 CUDA 用 GPU 0，否则 CPU（-1）
  pipe = pipeline(
    "text-generation",
    model=model_name,
    # device_map =“自动”，  # 原注释保留：曾考虑 device_map="auto"，当前用显式 device
    device= 0 if torch.cuda.is_available() else -1
  )

  # 把管线交还给调用方（后面会用 tokenizer.apply_chat_template + pipe(...)）
  return pipe


In [ ]:
# ========== 缓存管线 + 真正跑一次生成 ==========
# lru_cache：同一 model_name 最多缓存 2 个已加载管线，避免每次点按钮都重新下载/加载
from functools import lru_cache

# maxsize=2：最多同时记住 2 个模型的 pipeline（换模型时自动淘汰旧的）
@lru_cache(maxsize=2)
def get_pipeline(model_name: str):
    # 缓存未命中时才真正调用 load_pipeline
    return load_pipeline(model_name)

# Gradio 回调：根据 UI 参数生成合成数据文本
def run_generation(model_name: str, user_prompt: str, format: str, record_count: int):
    """Generate synthetic data from the selected model and parameters."""
    # 空输入时用默认英文 user prompt（字符串影响模型行为，保持原文）
    user_prompt = user_prompt.strip() or "Generate synthetic records with diverse, realistic data."
    # 将 UI「Plain Text」映射到 FORMAT_RULES 里的键「Raw Text」
    if format == "Plain Text":
        format = "Raw Text"

    try:
        # 取（或加载）选定模型的 pipeline
        pipe = get_pipeline(model_name)
        # 拼 system + user messages
        messages = build_messages(format, record_count, user_prompt)
        # 用该模型 tokenizer 的 chat 模板把 messages 变成单条 prompt 字符串
        prompt = pipe.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        # 采样生成：限制新 token、温度与 top_p；return_full_text=False 只要续写部分
        output = pipe(
            prompt,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            return_full_text=False
        )
        # pipeline 返回 list[dict]，取出 generated_text 并去首尾空白
        text = output[0]["generated_text"].strip()
        return text
    except Exception as e:
        # 加载失败 / 推理异常时把错误信息回显到 UI（文案保持英文前缀 Error:）
        return f"Error: {str(e)}"


In [ ]:
# ========== Gradio Blocks：合成数据生成器 UI ==========
# title / theme 字符串影响界面展示，保持原文（可运行英文）
with gr.Blocks(title="Synthetic Data Generator", theme=gr.themes.Soft()) as demo:
    # 标题与使用说明（Markdown 文案保持英文原样）
    gr.Markdown("## Synthetic Data Generator")
    gr.Markdown("Describe the data you want. Example: *Generate weather data with fields: country, state, temperature, humidity*")

    # 左右两栏：左侧输入控件，右侧输出框
    with gr.Row():
        with gr.Column(scale=1):
            # 用户描述「要生成什么数据」的多行文本框
            prompt_input = gr.Textbox(
                label="What data to generate",
                placeholder="e.g. Generate weather data with fields: country, state, temperature, humidity",
                lines=3
            )
            # 输出格式下拉；choices/value 字符串必须与后面映射逻辑一致
            format_dropdown = gr.Dropdown(
                choices=["JSON", "CSV", "Plain Text", "Python Code", "JSONL"],
                value="JSON",
                label="Output Format"
            )
            # 生成条数滑条：1～5，默认 3，步进 1
            record_slider = gr.Slider(1, 5, value=3, step=1, label="Number of Records (max 5)")
            # 模型下拉：选项来自前面的 models 列表；默认尽量选第 2 个
            model_dropdown = gr.Dropdown(
                choices=models,
                value=models[1] if len(models) > 1 else models[0],
                label="Model"
            )
            # 主按钮：点击后调用 run_generation
            generate_btn = gr.Button("Generate", variant="primary")

        with gr.Column(scale=1):
            # 展示模型输出的大文本框
            output_text = gr.Textbox(label="Generated Output", lines=16)

    # 绑定点击事件：inputs 顺序必须与 run_generation 形参一致
    generate_btn.click(
        run_generation,
        inputs=[
            model_dropdown,
            prompt_input,
            format_dropdown,
            record_slider,
        ],
        outputs=output_text
    )

# 启动本地 Gradio 服务（theme 参数保持原调用）
demo.launch(theme=gr.themes.Soft())
